# 02 Data Validation

This notebook validates the generated customer churn training dataset before feature engineering and model training.

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np

## 2. Locate and Load Dataset

The notebook expects the project structure to contain `data/train_data.csv`. It also tries multiple relative paths so it works from common notebook locations.

In [ ]:
candidate_paths = [
    "../data/train_data.csv",
    "../../data/train_data.csv",
    "data/train_data.csv",
    "./data/train_data.csv"
]

file_path = None
for path in candidate_paths:
    if os.path.exists(path):
        file_path = path
        break

if file_path is None:
    raise FileNotFoundError("train_data.csv not found. Please check the data folder path.")

df = pd.read_csv(file_path)

print("Data loaded successfully")
print("File path:", file_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

## 3. Dataset Shape and Columns

In [ ]:
print("Shape:", df.shape)
print("Columns:")
for col in df.columns:
    print("-", col)

## 4. Missing Values Check

In [ ]:
missing_values = df.isnull().sum()
missing_values

## 5. Duplicate Records Check

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

## 6. Data Types Check

In [ ]:
df.dtypes

## 7. Summary Statistics

In [ ]:
df.describe()

## 8. Categorical Value Checks

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 50)

## 9. Target Variable Check

In [ ]:
target_col = "churn"

if target_col in df.columns:
    print("Target value counts:")
    print(df[target_col].value_counts(dropna=False))
    print("\nTarget percentage distribution:")
    print((df[target_col].value_counts(normalize=True, dropna=False) * 100).round(2))
else:
    print("Target column not found:", target_col)

## 10. Numeric Range Validation

In [ ]:
range_checks = {}

if "age" in df.columns:
    range_checks["age_invalid"] = df[(df["age"] < 18) | (df["age"] > 100)].shape[0]

if "tenure_months" in df.columns:
    range_checks["tenure_months_invalid"] = df[df["tenure_months"] < 0].shape[0]

if "monthly_spend" in df.columns:
    range_checks["monthly_spend_invalid"] = df[df["monthly_spend"] < 0].shape[0]

if "total_transactions" in df.columns:
    range_checks["total_transactions_invalid"] = df[df["total_transactions"] < 0].shape[0]

if "avg_session_time" in df.columns:
    range_checks["avg_session_time_invalid"] = df[df["avg_session_time"] < 0].shape[0]

if "support_ticket_count" in df.columns:
    range_checks["support_ticket_count_invalid"] = df[df["support_ticket_count"] < 0].shape[0]

if "last_login_days" in df.columns:
    range_checks["last_login_days_invalid"] = df[df["last_login_days"] < 0].shape[0]

range_checks

## 11. Customer ID Validation

In [ ]:
if "customer_id" in df.columns:
    print("Unique customer IDs:", df["customer_id"].nunique())
    print("Total rows:", len(df))
    print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
else:
    print("customer_id column not found")

## 12. Final Validation Summary

In [ ]:
validation_summary = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "missing_values_total": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "target_classes": int(df["churn"].nunique()) if "churn" in df.columns else None,
    "range_issues_total": int(sum(range_checks.values()))
}

validation_summary

## 13. Validation Decision

In [ ]:
if (validation_summary["missing_values_total"] == 0 and
    validation_summary["duplicate_rows"] == 0 and
    validation_summary["range_issues_total"] == 0 and
    validation_summary["target_classes"] == 2):
    print("Validation passed. Dataset is ready for preprocessing / feature engineering.")
else:
    print("Validation failed. Please review the validation summary above.")